# 02 — Model Training & Evaluation (Baseline + Untuned Models)

**Author:** Rudolph Otoo  
**Date:** 2026  

---

## Objective

Train four models (Dummy, Logistic Regression, Random Forest, Gradient
Boosting) on a strictly stratified train/validation/test split, then
evaluate each on the held-out test set. No hyperparameter tuning is applied
here; this notebook establishes the **untuned** benchmark from which
notebook 03 refines.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import Paths, Settings
from src.data import process_data
from src.features import split_features_target
from src.models import build_pipeline, list_models
from src.evaluate import evaluate_model, aggregate_benchmark_tables
from src.visualize import (
    set_global_style,
    plot_roc_curves,
    plot_confusion_matrices,
)

set_global_style()

## 1. Data Splitting

A strict 60 / 20 / 20 train/validation/test split is applied with
stratification on the target label. This guarantees that class prevalence
is preserved across all three partitions, eliminating a common source of
evaluation bias.

In [ ]:
settings = Settings()
frame    = process_data()
splits   = split_features_target(frame, settings=settings)

X_train, X_val, X_test = splits["X_train"], splits["X_val"], splits["X_test"]
y_train, y_val, y_test = splits["y_train"], splits["y_val"], splits["y_test"]

print(f"Train  : {X_train.shape[0]:>4d} samples  (class 1 = {y_train.mean():.3f})")
print(f"Val    : {X_val.shape[0]:>4d} samples  (class 1 = {y_val.mean():.3f})")
print(f"Test   : {X_test.shape[0]:>4d} samples  (class 1 = {y_test.mean():.3f})")

## 2. Train Untuned Models

Each model is instantiated with its **default hyperparameters** and fit on
the training split only. The `Pipeline` wrapper ensures the `StandardScaler`
is fit on training data within each fold.

In [ ]:
untuned_pipelines = {}
for model_name in list_models():
    pipe = build_pipeline(model_name, random_seed=settings.random_seed)
    pipe.fit(X_train, y_train)
    untuned_pipelines[model_name] = pipe
    print(f"  ✓ {model_name} fit on {X_train.shape[0]} training samples.")

## 3. Evaluate on Held-out Test Set

The test set is touched **once and once only**, at this stage. Any decision
made based on this evaluation must be frozen; the tuned pipeline (notebook 03)
uses the validation split exclusively.

In [ ]:
untuned_results = {}
for name, pipe in untuned_pipelines.items():
    untuned_results[name] = evaluate_model(
        pipe, X_test, y_test, model_name=name
    )

untuned_table = aggregate_benchmark_tables(untuned_results)
print("\n── Untuned benchmark (test set) ──")
untuned_table

## 4. Visualisation: ROC Curves & Confusion Matrices

ROC-AUC curves give a threshold-independent summary of discriminative
ability; confusion matrices surface clinically meaningful false negatives


In [ ]:
plot_roc_curves(untuned_pipelines, X_test, y_test)
plot_confusion_matrices(untuned_pipelines, X_test, y_test)

## 5. Summary

The untuned Gradient Boosting and Random Forest models are expected to
significantly outperform the logistic and dummy baselines. The tuned pipeline
(notebook 03) will quantify the marginal gain from hyperparameter optimisation
over these defaults.